# Guia de leitura do notebook

Este notebook executa um pipeline de classificação de tumores mamários. As células são organizadas nas etapas abaixo; leia cada descrição antes de executar o bloco de código seguinte.

## 1. Ambiente e dados

As primeiras células importam pandas e NumPy para manipulação, Matplotlib e Seaborn para visualização, e scikit-learn para divisão, transformação, modelagem e avaliação. O CSV possui 569 registros, a variável `diagnosis` (B = benigno; M = maligno) e 30 medidas numéricas. `head()` e `tail()` apresentam amostras; `shape`, `columns` e `info()` verificam dimensões, nomes e tipos; `describe()` resume as distribuições por média, quartis, desvio-padrão, mínimo e máximo.

## 2. Qualidade e análise exploratória

A contagem de nulos identifica `Unnamed: 32` como uma coluna 100% vazia, e a contagem de duplicidades confirma que não há linhas repetidas. O gráfico de barras (`countplot`) mostra a frequência de B e M e evidencia o balanceamento moderado (357 benignos e 212 malignos). O heatmap de correlação usa as variáveis `_mean`: cada célula representa a correlação entre duas medidas, com cores e valores entre -1 e 1. Os histogramas com KDE mostram a forma e a sobreposição das distribuições de `radius_mean`, `texture_mean`, `concavity_mean` e `area_mean` por diagnóstico. Os boxplots mostram mediana, intervalo interquartil e valores extremos desses atributos em cada classe. O pairplot usa dispersões entre pares de variáveis e distribuições diagonais para verificar agrupamentos e separabilidade visual; por desempenho, usa amostra de até 200 linhas.

## 3. Preparação

São removidos o identificador `id` e a coluna vazia. `LabelEncoder` converte B em 0 e M em 1, tornando maligno a classe positiva. `X` contém 30 características e `y` contém o alvo. A divisão 80/20 com `stratify=y` preserva a proporção das classes. O `StandardScaler` aprende média e desvio-padrão somente no treino e aplica a mesma transformação ao teste, evitando vazamento de informação.

## 4. Modelos e métricas

A função de avaliação calcula accuracy (acertos totais), precision (confiabilidade das previsões positivas), recall (malignos encontrados), F1 (equilíbrio entre precision e recall), ROC-AUC (capacidade de ordenar as classes por probabilidade) e matriz de confusão. A Regressão Logística é um modelo linear probabilístico, usada como baseline interpretável e eficiente. O KNN classifica pelo voto dos vizinhos mais próximos, sendo útil para testar separação local; por isso exige escala. A Random Forest combina várias árvores treinadas com amostras e atributos diferentes, capturando relações não lineares e reduzindo a variância; é uma boa candidata para dados tabulares.

## 5. Escala robusta e PCA

O `RobustScaler` usa mediana e intervalo interquartil, reduzindo a influência de outliers. O PCA transforma as 30 variáveis correlacionadas em componentes ortogonais e mantém componentes suficientes para explicar 95% da variância; no experimento, 10 componentes preservam 95,66%. O gráfico de variância acumulada mostra no eixo x a quantidade de componentes e no eixo y a informação retida, com linha de referência em 95%. Sobre esses componentes são testados Regressão Logística, Random Forest, KNN, Árvore de Decisão e SVM. A Árvore de Decisão cria regras por divisões sucessivas; a SVM procura uma fronteira com maior margem; ambos ampliam a comparação para relações não lineares.

## 6. Comparação e interpretação

As tabelas e gráficos de barras finais ordenam os modelos pelas métricas. A matriz de confusão detalha falsos positivos e falsos negativos, especialmente importantes quando a classe maligna é a positiva. A lista `results` acumula os três modelos originais e os cinco modelos com PCA; por isso a comparação final reúne oito experimentos. A célula extra de visão computacional é apenas um título e não possui implementação.

In [ ]:
# Core data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing & pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Evaluation
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

Machine Learning para Classificação de Câncer de Mama

In [ ]:
pd.set_option('display.max_columns', None)

# Carregamento dos dados e amostras

In [ ]:
df = pd.read_csv("../data/breast-cancer-wisconsin-data.csv")

In [ ]:
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
df.tail()

In [ ]:
print("Formato:", df.shape)
print("Colunas:", list(df.columns))

In [ ]:
df.info()

In [ ]:
# Resumo estatístico
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
# Valores em falta / nulos
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({"Qtd. de Nulos": missing, "Nulos %": missing_pct})
missing_df = missing_df[missing_df["Qtd. de Nulos"] > 0].sort_values("Qtd. de Nulos", ascending=False)

print(f"Total de valores em nulos no dataset: {df.isnull().sum().sum()}")
missing_df


In [ ]:
# Linhas duplicadas
duplicate_count = df.duplicated().sum()
print(f"Número de linhas duplicadas: {duplicate_count}")

In [ ]:
# Distribuição por classes
class_counts = df["diagnosis"].value_counts()
print(class_counts)
print(f"\nClass balance: {(class_counts / class_counts.sum() * 100).round(1).to_dict()}")

plt.figure(figsize=(6, 5))
sns.countplot(data=df, x="diagnosis", order=["B", "M"])
plt.title("Target Class Distribution (B = Benign, M = Malignant)")
plt.xlabel("Diagnosis")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap (Mean Features)
mean_features = [c for c in df.columns if c.endswith("_mean")]

plt.figure(figsize=(10, 8))
corr = df[mean_features].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5, square=True)
plt.title("Correlation Heatmap — Mean Features")
plt.tight_layout()
plt.show()


In [ ]:
# Distribuições das características por diagnóstico
key_features = ["radius_mean", "texture_mean", "concavity_mean", "area_mean"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col in zip(axes.flatten(), key_features):
    sns.histplot(data=df, x=col, hue="diagnosis", kde=True, palette="viridis", element="step", ax=ax)
    ax.set_title(f"Distribution of {col.replace('_', ' ').title()} by Diagnosis")

plt.tight_layout()
plt.show()

In [ ]:
# Boxplots
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, col in zip(axes, key_features):
    sns.boxplot(data=df, x="diagnosis", y=col, order=["B", "M"], hue="diagnosis", palette="mako", ax=ax)
    ax.set_title(col.replace("_", " ").title())

plt.tight_layout()
plt.show()

In [ ]:
# Pairplot
pair_cols = ["radius_mean", "texture_mean", "concavity_mean", "area_mean", "diagnosis"]
sample_df = df[pair_cols].sample(min(200, len(df)), random_state=42)

sns.pairplot(sample_df, hue="diagnosis", corner=True,
             height=1.8, plot_kws={"alpha": 0.6, "s": 20})
plt.suptitle("Pairplot of Key Features by Diagnosis (sampled)", y=1.02)
plt.show()

In [ ]:
# Lindando os dados
df.drop(['Unnamed: 32','id'],  axis=1,inplace=True)

# Encode target: maligno = 1 (classe positiva), benigno = 0
label_encoder = LabelEncoder()
df["diagnosis"] = label_encoder.fit_transform(df["diagnosis"])  # B=0, M=1

print(f"Dataset shape (Current): {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
# Feature Engineering

X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]

print("Feature matrix X:", X.shape)
print("Target vector y :", y.shape)
print(f"Positive class (malignant) rate: {y.mean():.2%}")


In [ ]:
# Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]:,} rows  |  Malignant rate: {y_train.mean():.2%}")
print(f"Test set    : {X_test.shape[0]:,} rows  |  Malignant rate: {y_test.mean():.2%}")


In [ ]:
# Feature Scaling
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

print(" Features scaled. Example — before vs. after for 'area_mean':")
print(f"   Before: mean={X_train['area_mean'].mean():.2f}, std={X_train['area_mean'].std():.2f}")
print(f"   After : mean={X_train_scaled['area_mean'].mean():.2f}, std={X_train_scaled['area_mean'].std():.2f}")

In [ ]:
# Avaliação de modelos
results = []  # recolhe um dicionário por modelo

def evaluate_metrics(model, model_name, X_test, y_test, results_list=results):
    """
    Avalia um modelo já treinado e exibe suas métricas.
    """

    # Predições
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = y_pred

    # Métricas
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)

    # Salvar resultados
    results_list.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

    # Exibir resultados
    print(f"📌 {model_name}")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print("-" * 50)

## Treinamento dos modelos com StandardScaler

Agora os classificadores são ajustados no conjunto de treino padronizado e avaliados no teste. A Regressão Logística estima a probabilidade de malignidade por uma combinação linear das características e serve como baseline; o KNN decide pelo voto dos exemplos mais próximos, razão pela qual a escala é essencial; e a Random Forest agrega árvores de decisão para capturar relações não lineares e reduzir a instabilidade de uma árvore isolada. `max_iter=5000` ajuda a regressão a convergir e `n_estimators=100` define 100 árvores na floresta.

In [ ]:
# Model Training

# 1º Logistic Regression
log_reg = LogisticRegression(random_state=42, max_iter=5000)
log_reg.fit(X_train_scaled, y_train)

evaluate_metrics(log_reg, "Logistic Regression", X_test_scaled, y_test)

In [ ]:
# 2º Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)

evaluate_metrics(rf, "Random Forest", X_test_scaled, y_test)

In [ ]:
# K-Nearest Neighbors (KNN)
knn = KNeighborsClassifier()
knn.fit(X_train_scaled, y_train)

evaluate_metrics(knn, "K-Nearest Neighbors", X_test_scaled, y_test)

In [ ]:
# Model Comparison
comparison_df = pd.DataFrame(results)
comparison_df = comparison_df.sort_values("Accuracy", ascending=False).reset_index(drop=True)
comparison_df.index += 1

comparison_df.style.background_gradient(
    subset=["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"], cmap="Greens"
).format({"Accuracy": "{:.4f}", "Precision": "{:.4f}", "Recall": "{:.4f}",
          "F1 Score": "{:.4f}", "ROC-AUC": "{:.4f}"})

In [ ]:
#Metric Comparison Bar Charts
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics_list = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]

for ax, metric in zip(axes.flatten(), metrics_list):
    sorted_df = comparison_df.sort_values(metric, ascending=True)
    sns.barplot(data=sorted_df, x=metric, y="Model", hue="Model", palette="mako", legend=False, ax=ax)
    ax.set_title(f"{metric} by Model")
    ax.set_xlim(sorted_df[metric].min() - 0.05, 1.0)

axes.flatten()[-1].axis("off")  # unused 6th subplot
plt.tight_layout()
plt.show()

In [ ]:
log_pred = log_reg.predict(X_test_scaled)
log_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

rf_pred = rf.predict(X_test_scaled)
rf_proba = rf.predict_proba(X_test_scaled)[:, 1]

# Confusion Matrix
best_model_name = comparison_df.iloc[0]["Model"]
pred_lookup = {
    "Logistic Regression": log_pred, "Random Forest": rf_pred,
}
proba_lookup = {
    "Logistic Regression": log_proba, "Random Forest": rf_proba,
}
best_pred = pred_lookup[best_model_name]
best_proba = proba_lookup[best_model_name]

cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"])
plt.title(f"Confusion Matrix — {best_model_name}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

print(classification_report(y_test, best_pred, target_names=["Benign", "Malignant"]))


## Segunda configuração: RobustScaler

Esta etapa repete a transformação usando mediana e intervalo interquartil. Ao contrário da média e do desvio-padrão, essas estatísticas são menos influenciadas por outliers. O scaler é ajustado somente em `X_train` e depois aplicado em `X_test`, mantendo a separação correta entre treino e teste.

In [ ]:
# Normalização robusta baseada na mediana
robust_scaler = RobustScaler()

X_train_robust = pd.DataFrame(
    robust_scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_robust = pd.DataFrame(
    robust_scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Dados normalizados utilizando mediana/IQR.")
print("Treino:", X_train_robust.shape)
print("Teste :", X_test_robust.shape)

## Redução de dimensionalidade com PCA

O PCA transforma as 30 características correlacionadas em componentes ortogonais. `n_components=0.95` solicita a quantidade mínima de componentes que preserve pelo menos 95% da variância; o resultado é 10 componentes e 95,66% de variância explicada. O gráfico seguinte apresenta a variância acumulada por componente: quanto mais a curva se aproxima de 1, mais informação original é preservada. O PCA é ajustado no treino e apenas projetado no teste para evitar vazamento.

In [ ]:
pca = PCA(n_components=0.95, random_state=42)

X_train_pca = pca.fit_transform(X_train_robust)
X_test_pca = pca.transform(X_test_robust)

print("Número de componentes originais:", X_train_robust.shape[1])
print("Número de componentes após PCA:", X_train_pca.shape[1])

print(f"Variância explicada: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    np.cumsum(pca.explained_variance_ratio_),
    marker="o"
)

plt.axhline(
    y=0.95,
    linestyle="--"
)

plt.xlabel("Número de Componentes")
plt.ylabel("Variância Explicada Acumulada")
plt.title("PCA — Variância Explicada Acumulada")

plt.grid(True)
plt.show()

## Modelos treinados sobre os componentes principais

Os componentes `PC1` a `PC10` substituem as variáveis originais. A Regressão Logística verifica se uma fronteira linear continua suficiente após a compressão; a Random Forest e a Árvore de Decisão testam regras não lineares; o KNN verifica a vizinhança no espaço reduzido; e a SVM procura uma fronteira de máxima margem. A SVM usa `probability=True` para permitir ROC-AUC com probabilidades, embora o scikit-learn emita um aviso de depreciação futura dessa opção.

In [ ]:
log_reg_pca = LogisticRegression(
    random_state=42,
    max_iter=5000
)

# Convert PCA arrays to DataFrames with appropriate column names
num_components = X_train_pca.shape[1]
pca_columns = [f'PC{i+1}' for i in range(num_components)]

X_train_pca_df = pd.DataFrame(X_train_pca, columns=pca_columns, index=X_train.index)
X_test_pca_df = pd.DataFrame(X_test_pca, columns=pca_columns, index=X_test.index)

log_reg_pca.fit(
    X_train_pca_df,
    y_train
)

evaluate_metrics(
    log_reg_pca,
    "Logistic Regression + RobustScaler + PCA",
    X_test_pca_df,
    y_test
)


In [ ]:
rf_pca = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_pca.fit(
    X_train_pca_df,
    y_train
)

evaluate_metrics(
    rf_pca,
    "Random Forest + RobustScaler + PCA",
    X_test_pca_df,
    y_test
)

In [ ]:
knn_pca = KNeighborsClassifier()

knn_pca.fit(
    X_train_pca_df,
    y_train
)

evaluate_metrics(
    knn_pca,
    "KNN + RobustScaler + PCA",
    X_test_pca_df,
    y_test
)

In [ ]:
tree_pca = DecisionTreeClassifier(
    random_state=42
)

tree_pca.fit(
    X_train_pca_df,
    y_train
)

evaluate_metrics(
    tree_pca,
    "Decision Tree + RobustScaler + PCA",
    X_test_pca_df,
    y_test
)

In [ ]:
svm_pca = SVC(
    probability=True,
    random_state=42
)

svm_pca.fit(
    X_train_pca_df,
    y_train
)

evaluate_metrics(
    svm_pca,
    "SVM + RobustScaler + PCA",
    X_test_pca_df,
    y_test
)

## Comparação final dos experimentos

As células finais transformam a lista `results` em tabelas ordenadas e gráficos de barras horizontais. Cada barra representa a accuracy de um modelo; as tabelas também permitem comparar precision, recall, F1 e ROC-AUC. A matriz de confusão mostra acertos e erros por classe, com atenção especial aos falsos negativos malignos. Como `results` é acumulada, a comparação reúne os modelos originais e os modelos com RobustScaler + PCA. A seção de visão computacional ao final é apenas um marcador e não contém código implementado.

In [ ]:
comparison_pca = pd.DataFrame(results)

comparison_pca = comparison_pca.sort_values(
    "Accuracy",
    ascending=False
).reset_index(drop=True)

comparison_pca

In [ ]:
comparison_all = pd.DataFrame(results)

display(
    comparison_all[
        ["Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]
    ].sort_values("Accuracy", ascending=False)
)

In [ ]:
comparison_final = pd.DataFrame(results)

plt.figure(figsize=(14, 7))

sns.barplot(
    data=comparison_final,
    x="Accuracy",
    y="Model"
)

plt.title("Comparação de Accuracy — Modelos Originais vs. RobustScaler + PCA")
plt.xlabel("Accuracy")
plt.ylabel("Modelo")
plt.xlim(0.75, 1.0)

plt.tight_layout()
plt.show()

In [ ]:
pca_comparison = comparison_final[
    comparison_final["Model"].str.contains("PCA")
].copy()

pca_comparison = pca_comparison.sort_values(
    "Accuracy",
    ascending=False
)

display(
    pca_comparison[
        ["Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]
    ]
)

EXTRA - Visão computacional
